In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("data.csv", low_memory=False)
print(df.shape)

(3000040, 66)


In [2]:
# filter to used cars only
df = df[df['is_new'] == False].copy()
print(df.shape)

(1529003, 66)


In [3]:
# drop columns
drop_cols = [
    'combine_fuel_economy', 'is_certified', 'vehicle_damage_category',
    'bed', 'cabin', 'bed_height', 'bed_length', 'is_oemcpo', 'is_cpo',
    'vin', 'description', 'main_picture_url', 'listing_id', 'sp_id', 'sp_name',
    'trimId', 'transmission_display', 'wheel_system_display', 'power', 'dealer_zip',
    'is_new',
]
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)
print(df.shape)
print(df.columns.tolist())

(1529003, 45)
['back_legroom', 'body_type', 'city', 'city_fuel_economy', 'daysonmarket', 'engine_cylinders', 'engine_displacement', 'engine_type', 'exterior_color', 'fleet', 'frame_damaged', 'franchise_dealer', 'franchise_make', 'front_legroom', 'fuel_tank_volume', 'fuel_type', 'has_accidents', 'height', 'highway_fuel_economy', 'horsepower', 'interior_color', 'isCab', 'latitude', 'length', 'listed_date', 'listing_color', 'longitude', 'major_options', 'make_name', 'maximum_seating', 'mileage', 'model_name', 'owner_count', 'price', 'salvage', 'savings_amount', 'seller_rating', 'theft_title', 'torque', 'transmission', 'trim_name', 'wheel_system', 'wheelbase', 'width', 'year']


In [ ]:
# columns where the number comes first
num_cols = ['horsepower', 'torque', 'fuel_tank_volume', 'front_legroom',
            'back_legroom', 'height', 'width', 'length', 'wheelbase', 'maximum_seating']

for col in num_cols:
    df[col] = df[col].astype(str).str.replace(',', '').str.split().str[0]
    df[col] = pd.to_numeric(df[col], errors='coerce')

# engine displacement just has commas
df['engine_displacement'] = df['engine_displacement'].astype(str).str.replace(',', '')
df['engine_displacement'] = pd.to_numeric(df['engine_displacement'], errors='coerce')

# engine cylinders pull first number from strings 
df['engine_cylinders'] = df['engine_cylinders'].astype(str).str.extract(r'(\d+)')
df['engine_cylinders'] = pd.to_numeric(df['engine_cylinders'], errors='coerce')

df['owner_count'] = pd.to_numeric(df['owner_count'], errors='coerce')

In [5]:
# type fixes
df['has_accidents'] = df['has_accidents'].map({True: 1, False: 0, 'True': 1, 'False': 0})
df['frame_damaged'] = df['frame_damaged'].map({True: 1, False: 0, 'True': 1, 'False': 0})
df['salvage'] = df['salvage'].map({True: 1, False: 0, 'True': 1, 'False': 0})
df['theft_title'] = df['theft_title'].map({True: 1, False: 0, 'True': 1, 'False': 0})
df['fleet'] = df['fleet'].map({True: 1, False: 0, 'True': 1, 'False': 0})
df['isCab'] = df['isCab'].map({True: 1, False: 0, 'True': 1, 'False': 0})
df['franchise_dealer'] = df['franchise_dealer'].astype(int)

df['listed_date'] = pd.to_datetime(df['listed_date'], errors='coerce')
print('types fixed')

types fixed


In [ ]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({'nan': np.nan, '': np.nan, '--': np.nan})
print('strings cleaned')

strings cleaned


In [7]:
# row filtering
before = len(df)

# remove low price junk listings
df = df[df['price'] >= 1500].copy()
print(f"after price >= $1500: {len(df)} (removed {before - len(df)})")

# remove pre 1990
before = len(df)
df = df[df['year'] >= 1990].copy()
print(f"after year >= 1990: {len(df)} (removed {before - len(df)})")

# require key features for modeling
before = len(df)
key_cols = ['mileage', 'year', 'make_name', 'model_name', 'horsepower', 'body_type']
df.dropna(subset=key_cols, inplace=True)
print(f"after dropping rows missing key features: {len(df)} (removed {before - len(df)})")

after price >= $1500: 1527988 (removed 1015)
after year >= 1990: 1522717 (removed 5271)
after dropping rows missing key features: 1451970 (removed 70747)


In [ ]:
# car age at time of listing 
df['age'] = 2020 - df['year']
print(df['age'].describe())

count    1.451970e+06
mean     4.316077e+00
std      3.906444e+00
min     -1.000000e+00
25%      2.000000e+00
50%      3.000000e+00
75%      6.000000e+00
max      3.000000e+01
Name: age, dtype: float64


In [9]:
# final summary
print(f"final shape: {df.shape}")
print()
print("nulls remaining:")
print(df.isnull().sum().sort_values(ascending=False).head(20))
print()
print("price stats:")
print(df['price'].describe())

final shape: (1451970, 46)

nulls remaining:
franchise_make          535538
interior_color          217085
highway_fuel_economy    184711
city_fuel_economy       184711
torque                  122702
major_options            52544
back_legroom             44599
owner_count              35708
exterior_color           30566
seller_rating            24588
transmission             22475
engine_type              20066
engine_cylinders         20066
fuel_type                20031
front_legroom            13988
maximum_seating           4670
fuel_tank_volume          4650
height                    4639
length                    4633
width                     4629
dtype: int64

price stats:
count    1.451970e+06
mean     2.248788e+04
std      1.764945e+04
min      1.500000e+03
25%      1.370000e+04
50%      1.930500e+04
75%      2.828500e+04
max      3.299995e+06
Name: price, dtype: float64


In [10]:
df.head()

,back_legroom,body_type,city,city_fuel_economy,daysonmarket,engine_cylinders,engine_displacement,engine_type,exterior_color,fleet,...,seller_rating,theft_title,torque,transmission,trim_name,wheel_system,wheelbase,width,year,age
9,33.8,SUV / Crossover,San Juan,NaN,510,4.0,2000.0,I4,Blanco,0.0,...,3.000000,0.0,295.0,A,P300 R-Dynamic SE AWD,AWD,105.6,82.7,2020,0
10,NaN,Coupe,Guaynabo,NaN,1252,4.0,1700.0,I4,Red,0.0,...,NaN,0.0,258.0,A,Launch Edition Coupe RWD,RWD,93.7,73.5,2015,5
12,35.1,Sedan,Guaynabo,22.0,1233,6.0,3000.0,I6,Silver,0.0,...,NaN,0.0,330.0,A,340i xDrive Sedan AWD,AWD,110.6,80.0,2016,4
36,35.7,Sedan,Bay Shore,NaN,33,4.0,1600.0,I4,Phantom Black,0.0,...,3.447761,0.0,195.0,A,Sport Sedan FWD,FWD,106.3,70.9,2017,3
38,38.1,Sedan,Bay Shore,27.0,55,4.0,1500.0,I4,Silver Ice Metallic,1.0,...,3.447761,0.0,184.0,A,LT FWD,FWD,111.4,73.0,2018,2


In [11]:
df.to_csv("used_cars_cleaned.csv", index=False)
print("saved used_cars_cleaned.csv")

saved used_cars_cleaned.csv
